# 03 · Download the document corpus

> **Run order.** This notebook is step 3 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Fetches the annual report PDFs described by `configs/documents.yaml`.

**No PDF is ever committed.** `data/` is git-ignored; the manifest plus this
notebook reconstructs the corpus byte-for-byte, and SHA-256 makes that
reconstruction *verifiable* rather than merely hopeful.

Two sites in the corpus (`tcs.com`, `infosys.com`) return **HTTP 403** to any
scripted request — Akamai, even with full browser headers and a `Referer`. Those
are marked `fetch: manual` rather than worked around; the checksum still
guarantees everyone has identical bytes.

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from analyst.manifest import load_manifest

specs = load_manifest()
pd.DataFrame([
    {"ticker": s.ticker, "fy": s.fiscal_year, "fetch": s.fetch.value,
     "sha256": (s.sha256 or "")[:12] or "(not yet pinned)", "title": s.title}
    for s in specs
])

## Download

Behaviour worth knowing:
- file present **and** checksum matches → skipped
- file present, checksum **disagrees** → hard error. The source changed under us; that is never silently overwritten
- interrupted download → written to `.part`, renamed only on completion, so a truncated PDF never looks cached

In [ ]:
import os
import httpx
from sqlalchemy.dialects.postgresql import insert
from tenacity import retry, stop_after_attempt, wait_exponential

from analyst.config import get_settings
from analyst.db import session_scope
from analyst.manifest import DEFAULT_MANIFEST, FetchMode, pin_checksums, pin_key, sha256_file
from analyst.models import Document
from analyst.provenance import make_document_id

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"),
    "Accept": "application/pdf,*/*;q=0.8",
}
settings = get_settings()

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=20))
def download(spec, dest_tmp: str) -> None:
    with httpx.stream("GET", spec.url, headers=HEADERS, follow_redirects=True, timeout=120.0) as r:
        r.raise_for_status()
        if "pdf" not in r.headers.get("content-type", "").lower():
            raise ValueError(f"expected a PDF, got {r.headers.get('content-type')!r}")
        with open(dest_tmp, "wb") as fh:
            for chunk in r.iter_bytes(1 << 16):
                fh.write(chunk)

rows, pinned, manual = [], {}, []
for spec in specs:
    path = spec.local_path(settings.data_dir)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        digest = sha256_file(path)
        if spec.sha256 and digest != spec.sha256:
            raise SystemExit(f"CHECKSUM MISMATCH for {spec.filename}")
        status = "cached"
    elif spec.fetch is FetchMode.MANUAL:
        manual.append(spec)
        continue
    else:
        tmp = str(path) + ".part"
        download(spec, tmp)
        os.replace(tmp, path)
        digest = sha256_file(path)
        pinned[pin_key(spec)] = digest
        status = "downloaded"

    with session_scope() as s:
        values = {
            "document_id": make_document_id(spec.ticker, spec.doc_type, spec.fiscal_year, digest),
            "ticker": spec.ticker, "doc_type": spec.doc_type, "fiscal_year": spec.fiscal_year,
            "title": spec.title, "source_url": spec.url, "sha256": digest,
            "local_path": str(path), "size_bytes": path.stat().st_size,
        }
        stmt = insert(Document).values(**values)
        s.execute(stmt.on_conflict_do_update(
            constraint="uq_documents_identity",
            set_={k: v for k, v in values.items() if k != "document_id"},
        ))
    rows.append({"ticker": spec.ticker, "fy": spec.fiscal_year, "status": status,
                 "MB": round(path.stat().st_size / 1048576, 1), "sha": digest[:12]})

if pinned:
    pin_checksums(DEFAULT_MANIFEST, pinned)

print(f"available: {len(rows)}   manual downloads still pending: {len(manual)}")
for m in manual:
    print(f"  {m.ticker} FY{m.fiscal_year}:  open {m.url}")
    print(f"  {'':>12} save {m.local_path(settings.data_dir)}")
pd.DataFrame(rows)